# Hake School Region Browser

This notebook demonstrates how to use the interactive `region_browser` to view and edit acoustic regions on an echogram.

The region data follows the [Echoregions](https://echoregions.readthedocs.io/) **Regions2D** standard format, which is widely used in the fisheries acoustics community for polygon annotations of sonar data.

In [1]:
import xarray as xr
import fsspec
import pandas as pd
import numpy as np
import ast
import echoshader

# S3 Connection
S3_URL = "s3://agr230002-bucket01/hake_data/data_zarr/MVBS/2017/x0062_8_wt_20170731_180848_f0002.zarr"
storage_options = {"endpoint_url": "https://sdsc.osn.xsede.org"}
fs = fsspec.filesystem("s3", anon=True, client_kwargs=storage_options)

# Load data
ds_ooi = xr.open_zarr(fs.get_mapper(S3_URL), consolidated=False)

# Display the dataset structure
ds_ooi

<xarray.Dataset> Size: 6MB
Dimensions:            (channel: 3, ping_time: 344, depth: 759)
Coordinates:
  * channel            (channel) <U37 444B 'GPT  18 kHz 009072058c8d 1-1 ES18...
  * ping_time          (ping_time) datetime64[ns] 3kB 2017-07-31T18:08:45 ......
  * depth              (depth) float64 6kB 0.0 1.0 2.0 3.0 ... 756.0 757.0 758.0
Data variables:
    Sv                 (channel, ping_time, depth) float64 6MB dask.array<chunksize=(3, 344, 759), meta=np.ndarray>
    frequency_nominal  (channel) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    latitude           (ping_time) float64 3kB dask.array<chunksize=(344,), meta=np.ndarray>
    longitude          (ping_time) float64 3kB dask.array<chunksize=(344,), meta=np.ndarray>
Attributes:
    processing_function:          commongrid.compute_MVBS
    processing_level:             Level 3A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...
    processing_software_name:     echopype
    processing_software_version:  0.9.0
    processing_time:              2024-08-14T00:14:32Z

## Loading Region Data

### About Echoregions

[Echoregions](https://echoregions.readthedocs.io/) is a Python package that interfaces with water column sonar annotations. It provides tools to:

- Parse manual annotations from Echoview software
- Create training datasets for machine learning models
- Generate masks for biomass estimation
- Integrate seamlessly with Echopype xarray datasets

### The Regions2D Format

The **Regions2D** format stores each polygon region as a row with three columns:

- `region_id`: Unique identifier (integer)
- `time`: 1-D array of datetime64[ns] values defining polygon vertices
- `depth`: 1-D array of float values (in meters) for each vertex

This format allows our Region Browser to:
- Load and edit regions interactively
- Export to CSV for use with Echoregions tools
- Integrate with ML training pipelines
- Work with existing Echoview workflows

### Parsing the CSV

We load regions from `sample_hake_regions.csv` and parse the array strings into numpy arrays. While our Region Browser is compatible with the Echoregions format, it does not require Echoregions to be installed.

**Note:** If you have Echoregions installed, you can also load regions using:
```python
import echoregions as er
regions2d = er.read_regions_csv('sample_hake_regions.csv')
sample_df = regions2d.data[['region_id', 'time', 'depth']]
```

For this tutorial, we parse the CSV directly to show how the format works.

*You can also use the "Load CSV" button inside the browser to import files interactively!*

For more details on the Regions2D format, see the [Echoregions Regions2D documentation](https://echoregions.readthedocs.io/en/stable/regions2d.html).

In [2]:
# Load the CSV file
sample_df = pd.read_csv('sample_hake_regions.csv')

# Parse the string representations back into numpy arrays
# The CSV stores arrays as strings, so we convert them back to proper data types
sample_df['time'] = sample_df['time'].apply(
    lambda x: np.array(ast.literal_eval(x), dtype='datetime64[ns]')
)
sample_df['depth'] = sample_df['depth'].apply(
    lambda x: np.array(ast.literal_eval(x), dtype='float64')
)

# Display the parsed DataFrame
sample_df

,region_id,time,depth
0,1,"[2017-07-31T18:10:00.000000000, 2017-07-31T18:...","[40.0, 40.0, 80.0, 80.0]"
1,2,"[2017-07-31T18:20:00.000000000, 2017-07-31T18:...","[100.0, 100.0, 150.0, 150.0]"
2,3,"[2017-07-31T18:30:00.000000000, 2017-07-31T18:...","[50.0, 50.0, 90.0, 90.0]"
3,4,"[2017-07-31T18:12:00.000000000, 2017-07-31T18:...","[60.0, 60.0, 100.0, 100.0]"


In [ ]:
# Launch the Region Browser
browser = echoshader.region_browser(
    ds=ds_ooi,
    regions_df=sample_df,
    cache_backgrounds=True
)

browser.show()

Pre-caching backgrounds...
  Processed Region 1
  Processed Region 2
  Processed Region 3
  Processed Region 4
Cached 4 backgrounds successfully.
Launching server at http://localhost:60933


Reset Region 1 to baseline
Reset Region 1 to baseline
Reset Region 1 to baseline
Reset Region 1 to baseline
Reset Region 1 to baseline
Loaded 4 regions successfully.
Reset Region 1 to baseline


## Using the Browser

### Browse Mode
- Navigate between regions using the dropdown menu or Previous/Next buttons
- View the annotated regions overlaid on the echogram

### Edit Mode

**Editing Tools (in the right toolbar):**
- **Polygon Draw:** Click to add vertices, double-click last vertex to finish polygon. Drag to move entire polygon.
- **Polygon Edit:** Drag individual vertices to reposition them. Double-click a vertex to remove it.
- **Reset Region:** Restores the original polygon if you made a mistake.

**Workflow:**
1. Toggle to **Edit** mode using the mode selector
2. Select the appropriate editing tool from the right toolbar (PolyDraw or PolyEdit)
3. Make your edits to the polygon
4. Click **Apply Edit** to update the region in memory
5. Edit additional regions as needed
6. Click **Export to CSV** to save all changes to disk

⚠️ **Important:** Changes are only saved in memory until you export to CSV!

### Working with Echoregions

The exported CSV files are fully compatible with the Echoregions library. You can load them for further analysis:
```python
import echoregions as er

# Load regions
regions2d = er.read_regions_csv('edited_regions_20240315_120000.csv')

# Create masks for machine learning
mask_ds, region_points = regions2d.region_mask(
    ds_ooi["Sv"].isel(channel=1).drop_vars("channel"),
    region_class="Hake"
)

# Plot regions
regions2d.plot(region_class="Hake", close_regions=True)

# Select specific regions
hake_regions = regions2d.select_region(region_class="Hake")
```

For more information on Echoregions workflows, see the [Echoregions documentation](https://echoregions.readthedocs.io/).